<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/06.Dataset_Annotation_and_Labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 6 — Dataset Annotation and Labeling

*Notebook pendamping untuk* LLM Design Patterns

Model tidak belajar dari kebenaran, tetapi dari label yang kita berikan. Kalau `Apple Inc.` terlabel PERSON di ratusan contoh, model akan dengan tekun mempelajari bahwa perusahaan adalah orang. Bab ini membahas cara membuat anotasi, mengukur mutunya, dan menskalakan prosesnya.

Contoh di buku sendiri memberi bahan latihan yang pas: beberapa offset karakternya meleset, dan kodenya membuang anotasi yang meleset itu **tanpa pesan apa pun**. Kita akan menangkap kegagalan sunyi seperti itu, memperbaikinya, lalu mengukur seberapa sering ia terjadi. Urutannya: data NER untuk spaCy; label kata vs token subword BERT, termasuk melatih head klasifikasi token di GPU; metrik mutu dan crowdsourcing; lalu pra-anotasi, active learning, dan bias.

**Tentang bukti.** File ini dikirim tanpa output. Angka di narasi untuk bagian CPU (tokenizer, simulasi NumPy, scikit-learn) berasal dari run lokal kami dengan seed tetap; versi pustaka lain bisa menggeser digit terakhir. Bagian GPU belum kami jalankan di CUDA/Colab, jadi narasinya menjelaskan cara membaca hasil tanpa mengklaim angkanya. Semua "annotator" di simulasi bersifat sintetis; tidak ada hasil anotasi manusia atau crowd sungguhan di notebook ini.

## Setup

Sebagian besar bab ini berjalan di CPU: tokenisasi spaCy, simulasi annotator dengan NumPy, dan scikit-learn. GPU bekerja di dua tempat, yaitu fine-tuning head klasifikasi token BERT dan pra-anotasi dengan model NER transformer. Karena dua bagian itu memang butuh GPU, pemeriksaan CUDA dilakukan di awal: lebih baik berhenti sekarang daripada diam-diam berjalan lambat di CPU.

Sel pertama hanya memasang paket yang belum ada di runtime. `torch` sengaja tidak dipasang ulang karena runtime GPU Colab sudah menyediakannya.

In [1]:
import importlib
import importlib.util
import subprocess
import sys

REQUIRED = {"datasets": "datasets", "spacy": "spacy", "scikit-learn": "sklearn",
            "scipy": "scipy", "transformers": "transformers"}
missing = [pkg for pkg, module in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("memasang:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import spacy

if not spacy.util.is_package("en_core_web_sm"):
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    importlib.invalidate_caches()
print("paket siap")

paket siap


Sel berikut adalah gerbang GPU. Ia tidak berhenti di `torch.cuda.is_available()`: ia mengalokasikan matriks di GPU, mengalikannya dengan mixed precision, menunggu GPU selesai (`synchronize`), lalu memastikan hasilnya bukan NaN/inf. Tipe data mixed precision dipilih dari *compute capability*: T4 (7.5) tidak punya bf16 native sehingga memakai fp16, sedangkan GPU 8.0 ke atas memakai bf16. Tanpa CUDA, sel berhenti dengan instruksi; profil CPU hanya berjalan bila Anda sengaja mengubah `ALLOW_CPU = True`.

In [2]:
import contextlib
import gc
import os
import random
from pathlib import Path

import numpy as np
import torch

ALLOW_CPU = False  # ubah ke True HANYA bila sengaja menjalankan profil CPU yang dikurangi
ALLOW_CPU = ALLOW_CPU or os.environ.get("CH06_ALLOW_CPU") == "1"

SEED = 0
WORK = Path("ch06_work")
WORK.mkdir(exist_ok=True)


def functional_cuda_check():
    """Alokasi + komputasi + sinkronisasi nyata di GPU; berhenti keras bila gagal."""
    device = torch.device("cuda")
    major, minor = torch.cuda.get_device_capability(device)
    amp_dtype = torch.bfloat16 if major >= 8 else torch.float16
    a = torch.randn(2048, 2048, device=device)
    with torch.autocast("cuda", dtype=amp_dtype):
        b = a @ a.T
    torch.cuda.synchronize()
    if not torch.isfinite(b).all().item():
        raise RuntimeError("Uji matmul CUDA menghasilkan NaN/inf. Restart runtime atau ganti GPU.")
    free, total = torch.cuda.mem_get_info()
    print(f"GPU       : {torch.cuda.get_device_name(device)} (compute capability {major}.{minor})")
    print(f"memori    : {free / 2**30:.1f} GiB bebas dari {total / 2**30:.1f} GiB")
    print(f"dtype AMP : {amp_dtype}")
    del a, b
    torch.cuda.empty_cache()
    return device, amp_dtype


if torch.cuda.is_available():
    DEVICE, AMP_DTYPE = functional_cuda_check()
    PROFILE = "gpu"
elif ALLOW_CPU:
    DEVICE, AMP_DTYPE = torch.device("cpu"), None
    PROFILE = "cpu-reduced"
    print("PROFIL CPU (opt-in): fine-tuning BERT memakai langkah lebih sedikit; ini bukan hasil GPU.")
else:
    raise RuntimeError(
        "GPU CUDA tidak tersedia. Di Colab: Runtime > Change runtime type > T4 GPU, lalu Run all. "
        "Untuk sengaja memakai CPU, set ALLOW_CPU = True di sel ini.")

FT_STEPS = 40 if PROFILE == "gpu" else 20
FT_LR = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def autocast_ctx():
    """Mixed precision hanya di CUDA; profil CPU tetap fp32."""
    if DEVICE.type == "cuda":
        return torch.autocast("cuda", dtype=AMP_DTYPE)
    return contextlib.nullcontext()


print(f"DEVICE = {DEVICE} | PROFILE = {PROFILE} | FT_STEPS = {FT_STEPS}")

GPU       : Tesla T4 (compute capability 7.5)
memori    : 14.4 GiB bebas dari 14.6 GiB
dtype AMP : torch.float16
DEVICE = cuda | PROFILE = gpu | FT_STEPS = 40


In [3]:
import platform

import datasets
import sklearn
import transformers

nlp_sm = spacy.load("en_core_web_sm")
sm_ok = spacy.util.is_compatible_version(spacy.__version__, nlp_sm.meta["spacy_version"])
print("python       :", platform.python_version())
print("torch        :", torch.__version__, "| build CUDA:", torch.version.cuda)
print("transformers :", transformers.__version__)
print("spaCy        :", spacy.__version__, "| en_core_web_sm", nlp_sm.meta["version"], "| kompatibel:", sm_ok)
print("scikit-learn :", sklearn.__version__)
print("datasets     :", datasets.__version__)
assert sm_ok, "Versi en_core_web_sm tidak cocok dengan spaCy; jalankan ulang sel instalasi."

python       : 3.13.16
torch        : 2.11.0+cu130 | build CUDA: 13.0
transformers : 5.20.2
spaCy        : 3.8.16 | en_core_web_sm 3.8.0 | kompatibel: True
scikit-learn : 1.6.1
datasets     : 4.8.5


## Pentingnya anotasi berkualitas

Anotasi yang baik itu **konsisten** (kasus serupa diberi label serupa), **lengkap** (tidak ada entitas yang terlewat), dan **akurat** terhadap pedoman anotasi. Kendaraan bab ini adalah NER (*named entity recognition*): menandai potongan teks seperti nama orang, organisasi, atau tempat.

Untuk NER, annotator biasanya menyimpan entitas sebagai **offset karakter** `(start, end, label)`, dengan karakter di posisi `end` tidak ikut. Di kalimat `"Apple Inc. is planning ..."`, entitas `Apple Inc.` menempati karakter 0 sampai 10, jadi anotasinya `(0, 10, "ORG")`. spaCy mengubah offset itu menjadi objek `Span` lewat `doc.char_span`, dengan satu syarat: offset harus jatuh tepat di batas **token** spaCy. Kalau tidak, `char_span` mengembalikan `None`.

> **Listing buku: data NER yang kehilangan entitas tanpa pesan**
>
> ```python
> annotations = [[(0, 9, "ORG"), (41, 49, "GPE")],
>                [(0, 9, "ORG"), (14, 27, "PERSON")]]
> span = doc.char_span(start, end, label=label)
> if span:
>     ents.append(span)
> ```
>
> Listing ini berjalan tanpa error. Namun saat `train.spacy` dibaca balik, kalimat pertama tersimpan **tanpa entitas sama sekali**: `char_span` mengembalikan `None` untuk kedua offset-nya dan `if span:` membuangnya tanpa jejak. Sel berikut menjalankan offset yang sama lewat fungsi yang *melaporkan* setiap span yang ditolak.

In [4]:
from spacy.tokens import DocBin

nlp_blank = spacy.blank("en")  # hanya tokenizer; CPU


def create_training_data(texts, annotations, alignment_mode="strict"):
    """Seperti listing buku, tetapi span yang ditolak dicatat, bukan dibuang diam-diam."""
    db = DocBin()
    dropped = []
    for text, annot in zip(texts, annotations, strict=True):
        doc = nlp_blank.make_doc(text)
        ents = []
        for start, end, label in annot:
            span = doc.char_span(start, end, label=label, alignment_mode=alignment_mode)
            if span is None:
                dropped.append((text[start:end], start, end, label))
            else:
                ents.append(span)
        doc.ents = ents
        db.add(doc)
    for piece, start, end, label in dropped:
        print(f"DITOLAK: ({start}, {end}, {label}) -> {piece!r}")
    return db, dropped


ner_texts = [
    "Apple Inc. is planning to open a new store in New York.",
    "Microsoft CEO Satya Nadella announced new AI features.",
]
book_annotations = [
    [(0, 9, "ORG"), (41, 49, "GPE")],
    [(0, 9, "ORG"), (14, 27, "PERSON")],
]
_, dropped = create_training_data(ner_texts, book_annotations)
print(f"{len(dropped)} dari 4 span ditolak")

DITOLAK: (0, 9, ORG) -> 'Apple Inc'
DITOLAK: (41, 49, GPE) -> 'e in New'
2 dari 4 span ditolak


Dua span ditolak, dengan penyebab yang berbeda:

- `(0, 9)` menunjuk `'Apple Inc'`. Tokenizer spaCy memperlakukan `Inc.` (dengan titik) sebagai satu token di karakter 6–10, jadi offset yang berhenti di 9 membelah token. Isinya benar, **batasnya** yang salah.
- `(41, 49)` menunjuk `'e in New'`. Ini salah **isi**: `New York` sebenarnya ada di karakter 46–54.

Perbedaan itu penting karena spaCy menyediakan `alignment_mode` untuk menolong kesalahan batas: `"strict"` (default) menolak, `"expand"` melebar ke semua token yang tersentuh, `"contract"` menyusut ke token yang tercakup penuh. Lihat apa yang terjadi pada kedua offset buku:

In [5]:
doc0 = nlp_blank.make_doc(ner_texts[0])
for start, end in [(0, 9), (41, 49)]:
    results = {}
    for mode in ("strict", "expand", "contract"):
        span = doc0.char_span(start, end, alignment_mode=mode)
        results[mode] = None if span is None else span.text
    print(f"({start}, {end}) -> {results}")

(0, 9) -> {'strict': None, 'expand': 'Apple Inc.', 'contract': 'Apple'}
(41, 49) -> {'strict': None, 'expand': 'store in New', 'contract': 'in New'}


`expand` menyelamatkan `(0, 9)` menjadi `Apple Inc.`, tetapi mengubah `(41, 49)` menjadi `store in New`: entitas yang salah, dan kali ini tidak ditolak. Jadi `alignment_mode` hanya menolong kesalahan batas; kesalahan isi harus diperbaiki di sumbernya. Cara paling aman adalah menurunkan offset dari teks entitasnya sendiri, bukan menghitung karakter dengan tangan:

In [6]:
def spans_from_strings(text, entities):
    """Offset dari teks entitas; entitas yang muncul 0 atau >1 kali ditolak keras."""
    spans = []
    for piece, label in entities:
        if text.count(piece) != 1:
            raise ValueError(f"{piece!r} muncul {text.count(piece)} kali di {text!r}")
        start = text.index(piece)
        spans.append((start, start + len(piece), label))
    return spans


fixed_annotations = [
    spans_from_strings(ner_texts[0], [("Apple Inc.", "ORG"), ("New York", "GPE")]),
    spans_from_strings(ner_texts[1], [("Microsoft", "ORG"), ("Satya Nadella", "PERSON")]),
]
print(fixed_annotations)

db, dropped = create_training_data(ner_texts, fixed_annotations)
assert not dropped
db.to_disk(WORK / "train.spacy")

# baca balik file yang baru ditulis: jangan percaya sel yang diam
reloaded = list(DocBin().from_disk(WORK / "train.spacy").get_docs(nlp_blank.vocab))
for doc in reloaded:
    print([(ent.text, ent.label_) for ent in doc.ents])
assert sum(len(doc.ents) for doc in reloaded) == 4

[[(0, 10, 'ORG'), (46, 54, 'GPE')], [(0, 9, 'ORG'), (14, 27, 'PERSON')]]
[('Apple Inc.', 'ORG'), ('New York', 'GPE')]
[('Microsoft', 'ORG'), ('Satya Nadella', 'PERSON')]


Keempat entitas selamat, dan buktinya datang dari file yang dibaca balik. `spans_from_strings` sengaja menolak entitas yang muncul lebih dari sekali, karena pencarian string tidak tahu kemunculan mana yang dimaksud.

### Under the hood: seberapa rapuh offset karakter?

Kesalahan seperti `(0, 9)` sangat manusiawi: lupa titik, ikut menghitung spasi, atau alat anotasi memakai konvensi berbeda. Berapa banyak entitas hilang kalau offset meleset sedikit saja?

Eksperimennya: 6 kalimat dengan 17 entitas referensi yang kami tulis sendiri (bukan anotasi manusia independen). Kedua ujung setiap span digeser acak paling jauh ±1, ±2, atau ±3 karakter dan diulang 10 kali, sehingga tiap baris tabel berisi 170 percobaan. Pergeseran ini sintetis; kesalahan annotator sungguhan tidak tersebar seragam. Kolom terakhir menghitung hasil `expand` yang persis sama dengan teks entitas asli. CPU, seed 0.

In [7]:
REFERENCE = [
    ("Apple Inc. opened a research campus in Austin last spring.",
     [("Apple Inc.", "ORG"), ("Austin", "GPE"), ("last spring", "DATE")]),
    ("Chancellor Angela Merkel met President Emmanuel Macron in Berlin.",
     [("Angela Merkel", "PERSON"), ("Emmanuel Macron", "PERSON"), ("Berlin", "GPE")]),
    ("Goldman Sachs projected that Brazil would raise interest rates.",
     [("Goldman Sachs", "ORG"), ("Brazil", "GPE")]),
    ("Steve Wozniak and Steve Jobs founded Apple in Cupertino.",
     [("Steve Wozniak", "PERSON"), ("Steve Jobs", "PERSON"), ("Apple", "ORG"), ("Cupertino", "GPE")]),
    ("The New York Times reported a merger between Exxon Mobil and a rival.",
     [("The New York Times", "ORG"), ("Exxon Mobil", "ORG")]),
    ("Microsoft acquired a startup in Tel Aviv for $2 billion.",
     [("Microsoft", "ORG"), ("Tel Aviv", "GPE"), ("$2 billion", "MONEY")]),
]
ref_texts = [text for text, _ in REFERENCE]
ref_spans = [spans_from_strings(text, ents) for text, ents in REFERENCE]

# sebelum diganggu, semua span referensi harus pas batas token spaCy
for text, spans in zip(ref_texts, ref_spans):
    doc = nlp_blank.make_doc(text)
    assert all(doc.char_span(s, e) is not None for s, e, _ in spans), text

REPEATS = 10
rng = np.random.default_rng(0)
n_trials = REPEATS * sum(len(spans) for spans in ref_spans)
print(f"{len(ref_texts)} kalimat, {n_trials // REPEATS} entitas, x{REPEATS} ulangan = {n_trials} percobaan per baris")
print(f"{'geser maks':>10} | {'strict ditolak':>14} | {'expand ditolak':>14} | {'expand tepat':>12}")
for shift in (0, 1, 2, 3):
    lost_strict = 0
    lost_expand = 0
    exact_expand = 0
    for _ in range(REPEATS):
        for text, spans in zip(ref_texts, ref_spans):
            doc = nlp_blank.make_doc(text)
            for s, e, label in spans:
                s2 = max(0, s + int(rng.integers(-shift, shift + 1)))
                e2 = min(len(text), e + int(rng.integers(-shift, shift + 1)))
                strict = doc.char_span(s2, e2, label=label)
                expand = doc.char_span(s2, e2, label=label, alignment_mode="expand")
                lost_strict += strict is None
                lost_expand += expand is None
                exact_expand += expand is not None and expand.text == text[s:e]
    print(f"{'±' + str(shift) + ' char':>10} | {lost_strict / n_trials:>14.1%} | "
          f"{lost_expand / n_trials:>14.1%} | {exact_expand / n_trials:>12.1%}")

6 kalimat, 17 entitas, x10 ulangan = 170 percobaan per baris
geser maks | strict ditolak | expand ditolak | expand tepat
   ±0 char |           0.0% |           0.0% |       100.0%
   ±1 char |          80.0% |           0.0% |        89.4%
   ±2 char |          88.2% |           0.0% |        65.3%
   ±3 char |          92.4% |           0.0% |        52.9%


Baris ±0 adalah kontrol: tanpa gangguan, semua span diterima dan `expand` mengembalikan teks yang sama persis. Begitu kedua ujung boleh meleset satu karakter, `strict` menolak 80% span; dengan `if span:`, itu berarti 136 dari 170 entitas hilang tanpa pesan. Sebuah span hanya lolos `strict` bila kedua ujungnya kebetulan tetap di batas token: tidak bergeser (peluang 1/3 per ujung, 1/9 untuk keduanya) atau bergeser tepat ke batas token lain.

`expand` tidak pernah menolak, karena selalu bisa melebar ke batas token terdekat. Harganya terlihat di kolom terakhir: pada ±1, 89,4% hasilnya masih tepat; pada ±3 tinggal 52,9%, sisanya sudah menjadi teks lain, biasanya karena menyeret token tetangga seperti `store in New` tadi. Jadi `expand` berguna sebagai penyelamat, asalkan setiap span yang teksnya berubah dicatat untuk diperiksa.

## Strategi anotasi per jenis tugas

Satuan anotasi mengikuti tugasnya: **klasifikasi teks** melabeli seluruh teks, **NER** melabeli span di dalam teks, dan **question answering** (QA) melabeli posisi jawaban di dalam konteks. Kita mulai dari yang paling sederhana, klasifikasi dengan `datasets`. Bagian atas sel adalah listing buku; bagian bawah menambahkan satu kebiasaan yang layak dipakai sejak awal.

In [8]:
from datasets import ClassLabel, Dataset, Features, Value

cls_texts = [
    "This movie was fantastic!",
    "The service was terrible.",
    "The weather is nice today.",
]
cls_labels = [1, 0, 2]  # 1: positive, 0: negative, 2: neutral

dataset = Dataset.from_dict({"text": cls_texts, "label": cls_labels})
print(dataset[0])
print("tipe label tanpa ClassLabel:", dataset.features["label"])

features = Features({"text": Value("string"),
                     "label": ClassLabel(names=["negative", "positive", "neutral"])})
dataset = Dataset.from_dict({"text": cls_texts, "label": cls_labels}, features=features)
print("tipe label dengan ClassLabel:", dataset.features["label"])
print([dataset.features["label"].int2str(i) for i in dataset["label"]])

{'text': 'This movie was fantastic!', 'label': 1}
tipe label tanpa ClassLabel: Value('int64')
tipe label dengan ClassLabel: ClassLabel(names=['negative', 'positive', 'neutral'])
['positive', 'negative', 'neutral']


Listing buku menghasilkan `{'text': ..., 'label': 1}`, skema yang dipakai hampir semua alat Hugging Face. Namun tipe kolomnya hanya bilangan bulat: arti `1 = positive` hidup di komentar kode dan hilang begitu dataset disimpan. Dengan `ClassLabel`, nama kelas ikut tersimpan sebagai metadata dataset.

Naik satu tingkat: NER dengan skema **BIO**. `B-` menandai kata pertama sebuah entitas, `I-` kata lanjutannya, dan `O` kata di luar entitas. Dengan BIO, `New York Times` terbaca sebagai satu organisasi, sedangkan `Steve Jobs met Steve Wozniak` terbaca sebagai dua orang. Masalahnya, manusia melabeli **kata**, sedangkan BERT membaca **wordpiece** (potongan subword), dan satu kata bisa menjadi beberapa wordpiece.

> **Listing buku: label kata dipasangkan langsung ke wordpiece**
>
> ```python
> labels = ["B-ORG", "I-ORG", "O", "O", "O", "B-PER", "I-PER"]
> tokens = tokenizer.tokenize("Apple Inc. was founded by Steve Jobs")
> print(list(zip(tokens, labels)))
> ```
>
> Ada 7 label kata tetapi 8 wordpiece, karena `Inc.` pecah menjadi `inc` + `.`. Mulai dari titik itu semua label bergeser satu posisi: hasilnya memasangkan `('by', 'B-PER')` dan `('steve', 'I-PER')`, sedangkan `jobs` terbuang karena `zip` berhenti di daftar yang lebih pendek. Versi yang benar memakai `word_ids()` di sel berikut.

In [9]:
from transformers import AutoTokenizer

BERT_ID = "bert-base-uncased"
BERT_REV = "86b5e0934494bd15c9632b12f734a8a67f723594"  # commit Hub yang dipakai saat notebook disusun
tokenizer = AutoTokenizer.from_pretrained(BERT_ID, revision=BERT_REV)  # tokenizer saja; CPU

sentence = "Apple Inc. was founded by Steve Jobs"
words = sentence.split()
word_labels = ["B-ORG", "I-ORG", "O", "O", "O", "B-PER", "I-PER"]
assert len(words) == len(word_labels)
print(f"{len(words)} kata berlabel, {len(tokenizer.tokenize(sentence))} wordpiece")

enc = tokenizer(words, is_split_into_words=True)
pieces = tokenizer.convert_ids_to_tokens(enc["input_ids"])
prev_word = None
for piece, word_id in zip(pieces, enc.word_ids(), strict=True):
    if word_id is None:
        label = "-100 (token khusus, diabaikan loss)"
    elif word_id != prev_word:
        label = word_labels[word_id]                       # subword pertama membawa label kata
    else:
        label = word_labels[word_id].replace("B-", "I-")   # subword lanjutan: B- turun jadi I-
    print(f"{piece:<8} kata={str(word_id):<5} {label}")
    prev_word = word_id

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

7 kata berlabel, 8 wordpiece
[CLS]    kata=None  -100 (token khusus, diabaikan loss)
apple    kata=0     B-ORG
inc      kata=1     I-ORG
.        kata=1     I-ORG
was      kata=2     O
founded  kata=3     O
by       kata=4     O
steve    kata=5     B-PER
jobs     kata=6     I-PER
[SEP]    kata=None  -100 (token khusus, diabaikan loss)


Sekarang setiap wordpiece tahu kata asalnya. `inc` dan `.` sama-sama berasal dari kata indeks 1 (`Inc.`; indeks mulai dari 0), jadi keduanya `I-ORG`; `steve` dan `jobs` kembali menjadi orang. `[CLS]` dan `[SEP]` adalah token khusus BERT tanpa kata asal. Saat training, labelnya diisi `-100`, nilai yang diabaikan fungsi loss di PyTorch dan Hugging Face.

Dua detail menentukan hasil ini. Pertama, kata diberikan sudah terpisah (`is_split_into_words=True`) supaya indeks `word_ids()` sama dengan indeks daftar label. Kedua, untuk subword lanjutan ada dua kebijakan umum: mewarisi label sebagai `I-` (dipakai di sini) atau diberi `-100`. Keduanya sah asalkan sama antara training dan evaluasi.

Tugas ketiga adalah QA ekstraktif: anotasinya posisi awal dan akhir jawaban di dalam konteks. Listing buku:

In [10]:
context = "The capital of France is Paris. It is known for its iconic Eiffel Tower."
question = "What is the capital of France?"
answer = "Paris"

start_idx = context.index(answer)
end_idx = start_idx + len(answer)

print(f"Answer: {context[start_idx:end_idx]}")
print(f"Start index: {start_idx}, End index: {end_idx}")

Answer: Paris
Start index: 25, End index: 30


Hasilnya benar: `Paris` ada di karakter 25–30. Pola `context.index(answer)` mirip cara SQuAD menyimpan jawaban, yaitu `answer_start` plus teks jawabannya. Jebakannya muncul saat jawaban tertulis lebih dari sekali di konteks:

In [11]:
import re


def answer_spans(context, answer):
    """Semua posisi jawaban; lebih dari satu berarti perlu keputusan annotator."""
    return [(m.start(), m.end()) for m in re.finditer(re.escape(answer), context)]


context2 = "Paris hosted the 2024 Olympics. The capital of France is Paris."
print("semua kemunculan :", answer_spans(context2, "Paris"))
print("context.index()  :", context2.index("Paris"))

semua kemunculan : [(0, 5), (57, 62)]
context.index()  : 0


`index()` mengambil kemunculan pertama di karakter 0 (Paris sebagai tuan rumah Olimpiade), padahal jawaban untuk pertanyaan ibu kota ada di karakter 57. Karena itu anotasi QA yang serius bersifat *span-first*: annotator menandai posisi, lalu kode hanya memeriksa. Kalau anotasi dibuat otomatis dari string, setiap jawaban yang muncul lebih dari sekali, atau tidak muncul sama sekali, harus dikirim ke manusia.

## Tools dan platform anotasi

Untuk anotasi manual dalam jumlah besar, buku memetakan tiga alat: **Prodigy** (komersial, dari pembuat spaCy, kuat di active learning dan workflow berbasis Python), **Label Studio** (open source, multi-modal: teks, gambar, audio, video), dan **Doccano** (open source, khusus teks: sequence labeling, klasifikasi, seq2seq).

Notebook ini tidak membuka server atau akun anotasi apa pun. Kita cukup mensimulasikan file ekspor JSONL Doccano: satu objek JSON per baris, berisi `text` dan `labels`. Baris kedua adalah sampel buku apa adanya; baris pertama (komodo) adalah tambahan notebook ini.

In [12]:
import json

doccano_items = [
    {"text": "Komodo dragons are native to Indonesia.",
     "labels": [[0, 14, "ANIMAL"], [29, 38, "GPE"]]},
    {"text": "The majestic Bengal tiger prowled through the Sundarbans, a habitat it shares with spotted deer.",
     "labels": [[13, 25, "ANIMAL"], [47, 57, "GPE"], [81, 93, "ANIMAL"]]},  # sampel buku apa adanya
]
with open(WORK / "doccano_export.jsonl", "w") as f:
    for item in doccano_items:
        f.write(json.dumps(item) + "\n")
print((WORK / "doccano_export.jsonl").read_text())

{"text": "Komodo dragons are native to Indonesia.", "labels": [[0, 14, "ANIMAL"], [29, 38, "GPE"]]}
{"text": "The majestic Bengal tiger prowled through the Sundarbans, a habitat it shares with spotted deer.", "labels": [[13, 25, "ANIMAL"], [47, 57, "GPE"], [81, 93, "ANIMAL"]]}



Sebelum dipakai, periksa hal paling dasar: apakah setiap offset benar-benar menunjuk entitas yang dimaksud? Fungsi kecil di bawah mencetak `text[s:e]` dan memberi tanda bila span mulai atau berhenti di tengah kata, atau membawa spasi atau koma di tepinya.

In [13]:
def span_issues(text, start, end):
    piece = text[start:end]
    issues = []
    if start > 0 and text[start - 1].isalnum() and piece[:1].isalnum():
        issues.append("mulai di tengah kata")
    if end < len(text) and text[end].isalnum() and piece[-1:].isalnum():
        issues.append("berhenti di tengah kata")
    if piece != piece.strip():
        issues.append("spasi di tepi")
    if piece[-1:] in ",;:":
        issues.append("membawa tanda baca")
    return issues


def load_doccano_ner(file_path):
    with open(file_path) as f:
        return [json.loads(line) for line in f]


doccano_data = load_doccano_ner(WORK / "doccano_export.jsonl")
for item in doccano_data:
    for s, e, label in item["labels"]:
        issues = span_issues(item["text"], s, e)
        print(f"({s:>2}, {e:>2}, {label:<6}) -> {item['text'][s:e]!r:<16} {', '.join(issues) or 'OK'}")

( 0, 14, ANIMAL) -> 'Komodo dragons' OK
(29, 38, GPE   ) -> 'Indonesia'      OK
(13, 25, ANIMAL) -> 'Bengal tiger'   OK
(47, 57, GPE   ) -> 'undarbans,'     mulai di tengah kata, membawa tanda baca
(81, 93, ANIMAL) -> 'h spotted de'   mulai di tengah kata, berhenti di tengah kata


Baris komodo dan `Bengal tiger` bersih. `(47, 57)` menghasilkan `'undarbans,'`: meleset satu karakter dan menyeret koma. `(81, 93)` menghasilkan `'h spotted de'`: meleset dua karakter. Jadi dua dari tiga label di sampel buku tidak lolos ujian `text[s:e]`, ujian satu baris yang sebaiknya menjadi refleks setiap pemilik pipeline anotasi.

> **Listing buku: integrasi Doccano dengan trik *prefix-tokenize***
>
> ```python
> model = AutoModelForTokenClassification.from_pretrained("bert-base-uncased")
> ...
> start_token = len(tokenizer.tokenize(text[:start]))
> end_token = len(tokenizer.tokenize(text[:end]))
> ner_tags[start_token] = f'B-{label}'
> ```
>
> Kode ini berjalan, tetapi hasilnya salah dengan tenang. Dengan offset buku, `B-GPE` mendarat di `##dar` (bukan `sun`) dan koma ikut menjadi `I-GPE`. Model yang dimuat tidak pernah dipakai, dan head-nya masih bawaan `bert-base-uncased`: 2 label yang baru diinisialisasi, bukan skema BIO kita. Versi kerjanya: offset dikoreksi, diselaraskan lewat `offset_mapping`, lalu modelnya benar-benar dilatih di bagian setelah ini.

In [14]:
bengal = doccano_data[1]["text"]
intended = [("Bengal tiger", "ANIMAL"), ("Sundarbans", "GPE"), ("spotted deer", "ANIMAL")]
doccano_data[1]["labels"] = [list(span) for span in spans_from_strings(bengal, intended)]
print("offset terkoreksi:", doccano_data[1]["labels"])
for item in doccano_data:
    assert all(not span_issues(item["text"], s, e) for s, e, _ in item["labels"])


def char_spans_to_bio(text, spans):
    """Tag BIO per wordpiece lewat offset_mapping; span yang tak mengenai token = error."""
    enc = tokenizer(text, return_offsets_mapping=True, truncation=True)
    tags = ["O"] * len(enc["input_ids"])
    for s, e, label in spans:
        inside = [i for i, (ws, we) in enumerate(enc["offset_mapping"]) if we > ws and ws < e and we > s]
        if not inside:
            raise ValueError(f"span {text[s:e]!r} tidak mengenai token mana pun")
        if any(tags[i] != "O" for i in inside):
            raise ValueError(f"span {text[s:e]!r} bertumpuk dengan span lain")
        tags[inside[0]] = f"B-{label}"
        for i in inside[1:]:
            tags[i] = f"I-{label}"
    return enc, tags


enc, tags = char_spans_to_bio(bengal, doccano_data[1]["labels"])
pieces = tokenizer.convert_ids_to_tokens(enc["input_ids"])
for piece, (ws, we), tag in zip(pieces, enc["offset_mapping"], tags):
    if tag != "O" or piece == ",":
        print(f"{piece:<8} karakter {ws:>2}-{we:<2} {tag}")

offset terkoreksi: [[13, 25, 'ANIMAL'], [46, 56, 'GPE'], [83, 95, 'ANIMAL']]
bengal   karakter 13-19 B-ANIMAL
tiger    karakter 20-25 I-ANIMAL
sun      karakter 46-49 B-GPE
##dar    karakter 49-52 I-GPE
##ban    karakter 52-55 I-GPE
##s      karakter 55-56 I-GPE
,        karakter 56-57 O
spotted  karakter 83-90 B-ANIMAL
deer     karakter 91-95 I-ANIMAL


Tag kini mendarat tepat: keempat potongan `sun ##dar ##ban ##s` menjadi GPE, dan koma kembali `O`. `offset_mapping` memberi setiap wordpiece rentang karakternya di teks asli, jadi kita tidak perlu menebak posisi lewat panjang prefix. Kriteria `ws < e and we > s` berarti "wordpiece ini beririsan dengan span". Fungsi ini juga melempar error bila sebuah span tidak mengenai token mana pun atau bertumpuk dengan span lain, alih-alih membuangnya diam-diam.

### Dari label ke model: melatih head klasifikasi token di GPU

Listing buku memuat `AutoModelForTokenClassification` lalu berhenti. Bagian ini menyelesaikan niat itu dalam skala sekecil mungkin: dua kalimat Doccano yang sudah dikoreksi menjadi satu batch, tag BIO diubah menjadi angka, dan token khusus serta padding diberi `-100`. Tujuannya bukan model NER yang bagus. Kita ingin bukti bahwa label yang sudah diselaraskan benar-benar bisa dikonsumsi model: bentuk tensor cocok, semua tensor ada di device yang sama, dan loss turun.

In [15]:
from transformers import DataCollatorForTokenClassification

LABELS = ["O", "B-ANIMAL", "I-ANIMAL", "B-GPE", "I-GPE"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

train_features = []
for item in doccano_data:
    enc, tags = char_spans_to_bio(item["text"], item["labels"])
    label_ids = [-100 if ws == we else label2id[tag] for (ws, we), tag in zip(enc["offset_mapping"], tags)]
    train_features.append({"input_ids": enc["input_ids"],
                           "attention_mask": enc["attention_mask"],
                           "labels": label_ids})

collator = DataCollatorForTokenClassification(tokenizer)
batch = {name: tensor.to(DEVICE) for name, tensor in collator(train_features).items()}
n_target = int((batch["labels"] != -100).sum())
print({name: tuple(tensor.shape) for name, tensor in batch.items()})
print(f"token berlabel: {n_target} | device batch: {batch['input_ids'].device}")
assert n_target > 0

{'input_ids': (2, 23), 'attention_mask': (2, 23), 'labels': (2, 23)}
token berlabel: 30 | device batch: cuda:0


Model memuat bobot `bert-base-uncased` (sekitar 440 MB dari Hugging Face Hub, revisinya dikunci ke satu commit) dan memasang head baru berukuran 5 label. Peringatan bahwa bobot `classifier` baru diinisialisasi memang diharapkan, karena head itu belum pernah melihat data. Head acak memberi peluang hampir rata ke 5 label, sehingga loss awal seharusnya dekat ln 5 ≈ 1,61. Bobot disimpan dalam fp32, sedangkan komputasi maju memakai autocast: fp16 di T4, dengan `GradScaler` supaya gradien kecil tidak hilang, atau bf16 di GPU Ampere ke atas.

In [16]:
from transformers import AutoModelForTokenClassification

torch.manual_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(
    BERT_ID, revision=BERT_REV, num_labels=len(LABELS), id2label=id2label, label2id=label2id,
).to(DEVICE, dtype=torch.float32)
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)

param = next(model.parameters())
assert param.device.type == DEVICE.type and param.dtype == torch.float32
assert all(tensor.device == param.device for tensor in batch.values())
print(f"model di {param.device} | bobot {param.dtype} | autocast {AMP_DTYPE} | {FT_STEPS} langkah")

model.train()
losses = []
for step in range(FT_STEPS):
    optimizer.zero_grad(set_to_none=True)
    with autocast_ctx():
        loss = model(**batch).loss
    if not torch.isfinite(loss):
        raise RuntimeError(f"loss tidak finite di langkah {step}")
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    losses.append(loss.item())
    if step % 10 == 0 or step == FT_STEPS - 1:
        print(f"langkah {step:>2}: loss {losses[-1]:.3f}")

assert losses[-1] < 0.5 * losses[0], "loss tidak turun cukup jauh: periksa label, device, atau optimizer"

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

model di cuda:0 | bobot torch.float32 | autocast torch.float16 | 40 langkah
langkah  0: loss 1.805
langkah 10: loss 0.086
langkah 20: loss 0.010
langkah 30: loss 0.003
langkah 39: loss 0.002


In [17]:
model.eval()
with torch.no_grad(), autocast_ctx():
    logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
pred = logits.float().argmax(dim=-1).cpu()
gold = batch["labels"].cpu()
mask = gold != -100
train_acc = (pred[mask] == gold[mask]).float().mean().item()
print(f"akurasi token pada 2 kalimat latih: {train_acc:.1%} dari {int(mask.sum())} token")

pieces = tokenizer.convert_ids_to_tokens(batch["input_ids"][1].cpu().tolist())
for piece, g, p in zip(pieces, gold[1].tolist(), pred[1].tolist()):
    if g != -100 and (g != 0 or p != 0):
        print(f"{piece:<8} label {id2label[g]:<9} prediksi {id2label[p]}")

del model, optimizer, scaler, logits, batch
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

akurasi token pada 2 kalimat latih: 100.0% dari 30 token
bengal   label B-ANIMAL  prediksi B-ANIMAL
tiger    label I-ANIMAL  prediksi I-ANIMAL
sun      label B-GPE     prediksi B-GPE
##dar    label I-GPE     prediksi I-GPE
##ban    label I-GPE     prediksi I-GPE
##s      label I-GPE     prediksi I-GPE
spotted  label B-ANIMAL  prediksi B-ANIMAL
deer     label I-ANIMAL  prediksi I-ANIMAL


Loss yang turun jauh dari sekitar 1,6 dan akurasi token yang tinggi pada **dua kalimat latih** hanya membuktikan bahwa pipa datanya benar: label, padding, `-100`, device, dan optimizer bekerja bersama. Ini hafalan, bukan kemampuan NER. Seandainya label masih bergeser seperti di listing buku, model akan menghafal label yang salah dengan sama rajinnya dan loss tetap turun. Karena itu pemeriksaan `text[s:e]` dan penyelarasan harus dilakukan *sebelum* training.

## Mengelola kualitas anotasi

Mutu anotasi diukur dari beberapa sudut. **Inter-annotator agreement** (IAA) mengukur kesepakatan antar-annotator: Cohen's kappa untuk dua orang, Fleiss' kappa untuk banyak orang, Krippendorff's alpha untuk tipe data campuran atau data yang hilang. **Gold standard** mengukur akurasi terhadap label pakar. Untuk data timpang seperti NER, precision, recall, dan F1 lebih jujur daripada akurasi; anotasi kontinu seperti skor rating diukur dengan RMSE atau MAE; waktu per anotasi membantu menemukan item yang ambigu.

Listing buku untuk Cohen's kappa:

In [18]:
from sklearn.metrics import cohen_kappa_score

annotator1 = [0, 1, 2, 0, 1]
annotator2 = [0, 1, 1, 0, 1]

kappa = cohen_kappa_score(annotator1, annotator2)
print(f"Cohen's Kappa: {kappa}")

Cohen's Kappa: 0.6666666666666667


Kedua annotator sepakat di 4 dari 5 item, tetapi kappa hanya 0,67. Kappa memotong bagian kesepakatan yang bisa terjadi karena kebetulan. Hitungannya kecil:

In [19]:
a1 = np.array(annotator1)
a2 = np.array(annotator2)
p_o = (a1 == a2).mean()
p_e = sum((a1 == k).mean() * (a2 == k).mean() for k in np.union1d(a1, a2))
print(f"p_o = {p_o:.2f} | p_e = {p_e:.2f} | kappa = (p_o - p_e) / (1 - p_e) = {(p_o - p_e) / (1 - p_e):.3f}")

p_o = 0.80 | p_e = 0.40 | kappa = (p_o - p_e) / (1 - p_e) = 0.667


Kesepakatan teramati p_o = 0,80. Kesepakatan karena kebetulan, p_e, dihitung dari seberapa sering masing-masing annotator memakai tiap kelas. Annotator 1 memakai kelas 0, 1, 2 dengan proporsi 0,4 / 0,4 / 0,2; annotator 2 dengan 0,4 / 0,6 / 0. Kalau keduanya menebak sesuai kebiasaan masing-masing, mereka akan sepakat 0,4×0,4 + 0,4×0,6 + 0,2×0 = 0,40 dari waktu. Kappa = (0,80 − 0,40) / (1 − 0,40) = 0,67: dari ruang 60 poin persentase di atas kebetulan, mereka mengisi 40 poin.

Listing kedua buku membandingkan anotasi dengan gold standard. Di contoh ini "gold" hanyalah daftar contoh, bukan label pakar sungguhan:

In [20]:
def calculate_accuracy(gold_standard, annotations):
    return sum(
        g == a for g, a in zip(
            gold_standard, annotations
        )
    ) / len(gold_standard)

gold_standard = [0, 1, 2, 0, 1]
annotator_result = [0, 1, 1, 0, 1]

accuracy = calculate_accuracy(gold_standard, annotator_result)
print(f"Accuracy: {accuracy}")

Accuracy: 0.8


Hasilnya 0,8, sama dengan kesepakatan mentah tadi. Selisih 0,8 vs 0,67 terlihat kecil di sini, tetapi di data NER bisa sangat besar. Sebagian besar token NER berlabel `O`, dan pada data timpang kesepakatan tinggi bisa diperoleh tanpa membaca teks sama sekali.

### Under the hood: kesepakatan mentah vs kappa pada data timpang

Simulasi berikut membandingkan dua pasang annotator sintetis pada 20.000 token biner (1 = bagian entitas). Pasangan pertama adalah **penebak**: tidak membaca teks, hanya menjawab "entitas" dengan peluang sebesar proporsi entitas. Pasangan kedua adalah annotator yang **benar 90%** per token dengan kesalahan independen. Proporsi token `O` dinaikkan dari 50% sampai 99%. CPU, seed 1.

In [21]:
rng = np.random.default_rng(1)
N = 20_000

print(f"{'token O':>7} | {'penebak: akurasi':>16} {'sepakat':>8} {'kappa':>6} | {'annotator 90%: sepakat':>22} {'kappa':>6}")
for p_entity in (0.50, 0.20, 0.05, 0.01):
    truth = (rng.random(N) < p_entity).astype(int)           # 1 = token entitas
    guess1 = (rng.random(N) < p_entity).astype(int)           # menebak sesuai proporsi, tanpa membaca
    guess2 = (rng.random(N) < p_entity).astype(int)
    real1 = np.where(rng.random(N) < 0.9, truth, 1 - truth)   # benar 90%, salah independen
    real2 = np.where(rng.random(N) < 0.9, truth, 1 - truth)
    print(f"{1 - p_entity:>7.0%} | {(guess1 == truth).mean():>16.1%} {(guess1 == guess2).mean():>8.1%} "
          f"{cohen_kappa_score(guess1, guess2):>6.2f} | {(real1 == real2).mean():>22.1%} "
          f"{cohen_kappa_score(real1, real2):>6.2f}")

token O | penebak: akurasi  sepakat  kappa | annotator 90%: sepakat  kappa
    50% |            50.0%    49.7%  -0.01 |                  81.8%   0.64
    80% |            67.8%    68.2%   0.00 |                  82.4%   0.54
    95% |            90.4%    90.4%  -0.01 |                  82.3%   0.26
    99% |            98.0%    97.9%  -0.01 |                  82.0%   0.05


Lihat baris 95% `O`, proporsi yang wajar untuk token NER. Dua penebak yang tidak pernah membaca teks sepakat 90,4%, dan akurasinya terhadap kebenaran juga 90,4%. Dua annotator yang benar 90% hanya sepakat 82,3%. Menurut kesepakatan mentah, penebak tampak lebih baik.

Kappa membalik urutan itu: penebak mendapat −0,01 (tidak lebih baik dari kebetulan), annotator sungguhan 0,26. Kesepakatan para penebak memang persis yang diprediksi kebetulan untuk dua orang yang hampir selalu menjawab `O`: 0,95² + 0,05² ≈ 0,905.

Kolom kanan memberi pelajaran kedua. Kesepakatan mentah annotator 90% tetap sekitar 82% di semua baris (0,9² + 0,1² = 0,82), tetapi kappa mereka turun dari 0,64 ke 0,05 saat data makin timpang. Pada 99% `O` hampir semua token tidak menguji apa-apa, sehingga metrik per token sulit membedakan annotator kompeten dari kebetulan. Itulah sebabnya NER biasanya diukur dengan F1 per **entitas**, bukan akurasi per token.

## Crowdsourcing

Platform seperti Amazon Mechanical Turk memberi akses ke banyak annotator sekaligus, dengan mutu per orang yang tidak terjamin. Pertahanan standarnya: setiap item dilabeli beberapa orang, lalu jawabannya digabung. Listing buku memakai majority vote:

In [22]:
from collections import Counter

def aggregate_annotations(annotations):
    return Counter(annotations).most_common(1)[0][0]

crowd_annotations = [
    ['PERSON', 'PERSON', 'ORG', 'PERSON'],
    ['PERSON', 'ORG', 'ORG', 'PERSON'],
    ['PERSON', 'PERSON', 'ORG', 'LOC']
]

aggregated = [aggregate_annotations(annot)
              for annot in zip(*crowd_annotations)]
print(f"Aggregated annotations: {aggregated}")

Aggregated annotations: ['PERSON', 'PERSON', 'ORG', 'PERSON']


`zip(*crowd_annotations)` membalik matriks annotator × item menjadi daftar suara per item, lalu suara terbanyak menang. Di contoh buku setiap item punya pemenang yang jelas. Bagaimana kalau seri?

In [23]:
def aggregate_or_flag(votes):
    """Majority vote, tetapi suara seri dikirim ke adjudikasi alih-alih diputus urutan."""
    ranked = Counter(votes).most_common()
    if len(ranked) > 1 and ranked[0][1] == ranked[1][1]:
        return "PERLU_ADJUDIKASI"
    return ranked[0][0]


tied = ["ORG", "PERSON", "PERSON", "ORG"]
print("listing buku     :", aggregate_annotations(tied))
print("dengan penanda   :", aggregate_or_flag(tied))

listing buku     : ORG
dengan penanda   : PERLU_ADJUDIKASI


Pada suara 2–2, `most_common` memilih `ORG` hanya karena `ORG` muncul lebih dulu di daftar. Hasilnya deterministik secara teknis tetapi sembarang secara makna. Versi kedua menandai item seperti itu untuk diputus annotator senior.

Pertanyaan berikutnya lebih mendasar: berapa annotator per item yang cukup?

### Under the hood: aritmetika majority vote

Kalau setiap annotator benar dengan peluang p dan kesalahannya independen, akurasi majority vote dari k annotator bisa dihitung dengan distribusi binomial; inilah teorema juri Condorcet. Simulasi di bawah memakai 20.000 item biner dan membandingkan hasil terukur dengan rumusnya. Bagian kedua menguji tim campuran: satu ahli (benar 90%) dan dua annotator lemah (60%). Bobot log-odds di situ memakai akurasi **sebenarnya** tiap annotator, yang di dunia nyata tidak diketahui; anggap baris itu kontrol *oracle*, batas atas bila reliabilitas diketahui sempurna. CPU, seed 2.

In [24]:
from scipy.stats import binom

rng = np.random.default_rng(2)
N = 20_000

print("akurasi majority vote, terukur|teori")
print(f"{'p annotator':>11} | " + " | ".join(f"{'k=' + str(k):^11}" for k in (1, 3, 5, 9)))
for p in (0.55, 0.65, 0.75, 0.85):
    row = []
    for k in (1, 3, 5, 9):
        votes = rng.random((N, k)) < p                      # True = suara benar
        measured = (votes.sum(axis=1) * 2 > k).mean()
        theory = 1 - binom.cdf(k // 2, k, p)
        row.append(f"{measured:.3f}|{theory:.3f}")
    print(f"{p:>11.0%} | " + " | ".join(f"{cell:>11}" for cell in row))

ps = np.array([0.9, 0.6, 0.6])
votes = rng.random((N, 3)) < ps
majority = (votes.sum(axis=1) * 2 > 3).mean()
majority_theory = ps.prod() + sum(ps[[j for j in range(3) if j != i]].prod() * (1 - ps[i]) for i in range(3))
weights = np.log(ps / (1 - ps))                              # oracle: memakai p sebenarnya
weighted = (((votes * 2 - 1) @ weights) > 0).mean()
expert_alone = votes[:, 0].mean()
print()
print(f"bobot log-odds: {np.round(weights, 2)}")
print(f"tim [0.90, 0.60, 0.60]: majority {majority:.3f} (teori {majority_theory:.3f}) | "
      f"berbobot oracle {weighted:.3f} | ahli sendirian {expert_alone:.3f}")

akurasi majority vote, terukur|teori
p annotator |     k=1     |     k=3     |     k=5     |     k=9    
        55% | 0.551|0.550 | 0.574|0.575 | 0.587|0.593 | 0.625|0.621
        65% | 0.649|0.650 | 0.720|0.718 | 0.761|0.765 | 0.829|0.828
        75% | 0.747|0.750 | 0.840|0.844 | 0.897|0.896 | 0.951|0.951
        85% | 0.847|0.850 | 0.940|0.939 | 0.973|0.973 | 0.994|0.994

bobot log-odds: [2.2  0.41 0.41]
tim [0.90, 0.60, 0.60]: majority 0.787 (teori 0.792) | berbobot oracle 0.901 | ahli sendirian 0.901


Simulasi menempel rumus binomial dengan selisih beberapa perseribu. Selama tiap annotator lebih baik dari lempar koin dan kesalahannya independen, menambah orang menaikkan akurasi gabungan; annotator 75% menjadi 0,897 bila berlima. Imbal hasilnya cepat menurun. Untuk p = 75%, dua orang tambahan pertama (k=1 ke k=3) menambah 9,3 poin persentase, sedangkan empat orang tambahan dari k=5 ke k=9 hanya menambah 5,4 poin, padahal biaya naik sebanding jumlah orang. Itu sebabnya 3–5 annotator per item adalah praktik umum. Asumsi independen juga rapuh: annotator sungguhan cenderung salah di item sulit yang sama, dan kesalahan yang berkorelasi tidak terkoreksi oleh voting.

Pada tim campuran, majority vote polos jatuh ke 0,787 (rumus: 0,792), di bawah ahli yang bekerja sendirian, karena dua annotator lemah bisa bersekutu mengalahkannya. Bobot log-odds memberi ahli 2,20 dan tiap annotator lemah 0,41. Karena 2,20 lebih besar dari 0,41 + 0,41, keputusan berbobot selalu mengikuti ahli, sehingga akurasinya persis sama dengan akurasi ahli di sampel ini. Jadi pembobotan tidak melampaui ahli; ia mencegah informasi ahli terbuang. Di dunia nyata bobotnya harus diestimasi, misalnya dari item gold atau model Dawid-Skene, dan hasilnya akan berada di bawah batas oracle ini.

## Anotasi semi-otomatis

Arah sebaliknya: model memberi anotasi awal, manusia memeriksa dan mengoreksi. Listing buku memakai `en_core_web_sm`, pipeline spaCy kecil berbasis CNN yang dirancang untuk CPU. Untuk satu kalimat GPU tidak membantu, jadi listing ini memang berjalan di CPU.

In [25]:
import spacy

nlp = spacy.load("en_core_web_sm")

def semi_automated_ner(text):
    doc = nlp(text)
    return [(ent.start_char, ent.end_char, ent.label_)
            for ent in doc.ents]

text = "Apple Inc. was founded by Steve Jobs in Cupertino."
auto_annotations = semi_automated_ner(text)
print(f"Auto-generated annotations: {auto_annotations}")
# Human annotator would then verify and correct these annotations

Auto-generated annotations: [(0, 10, 'ORG'), (26, 36, 'PERSON'), (40, 49, 'GPE')]


Tiga entitas, dan offset pertamanya `(0, 10)` sudah termasuk titik di `Apple Inc.`: model spaCy dengan sendirinya menghormati batas token yang dilanggar listing pembuka. Memeriksa saran jauh lebih cepat daripada menandai dari nol, dan itulah dasar fitur pre-labeling di Prodigy dan Label Studio. Risikonya juga punya nama, **automation bias**: annotator yang lelah cenderung menyetujui saran yang salah.

Dalam praktik, saran model jarang diterima mentah-mentah. Biasanya ada aturan **routing**: saran yang meragukan dikirim ke manusia lebih dulu. Sel berikut menjalankan model NER transformer `dslim/bert-base-NER` (BERT-base yang di-fine-tune pada CoNLL-2003, dengan label PER/ORG/LOC/MISC) di GPU pada 6 kalimat dari eksperimen offset. Model diunduh dari Hugging Face Hub (sekitar 430 MB, tanpa login, revisi dikunci).

In [26]:
from transformers import pipeline

NER_MODEL_ID = "dslim/bert-base-NER"
NER_REV = "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc"  # commit Hub yang dipakai saat notebook disusun
INFER_DTYPE = AMP_DTYPE if DEVICE.type == "cuda" else torch.float32

ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_ID, revision=NER_REV)
ner_model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_ID, revision=NER_REV)
ner_model = ner_model.to(DEVICE, dtype=INFER_DTYPE).eval()
ner = pipeline("token-classification", model=ner_model, tokenizer=ner_tokenizer,
               aggregation_strategy="simple", device=DEVICE)
assert ner.model.device.type == DEVICE.type
print(f"model pra-anotasi di {ner.model.device} | dtype {next(ner.model.parameters()).dtype}")

bert_suggestions = ner(ref_texts, batch_size=8)
print(f"{sum(len(s) for s in bert_suggestions)} saran entitas dari {len(ref_texts)} kalimat")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model pra-anotasi di cuda:0 | dtype torch.float16
15 saran entitas dari 6 kalimat


Setiap saran sekarang diuji tiga aturan. Saran ditandai untuk review bila skornya di bawah 0,90, bila spaCy tidak mengusulkan span yang persis sama, atau bila span-nya tidak pas batas token spaCy (masalah yang sama dengan listing pembuka). Kolom terakhir membandingkan dengan 17 span referensi dari eksperimen offset.

In [27]:
SCORE_MIN = 0.90
rows = []
for i, (text, suggestions, doc_sm) in enumerate(zip(ref_texts, bert_suggestions, nlp_sm.pipe(ref_texts), strict=True)):
    tokens_doc = nlp_blank.make_doc(text)
    sm_spans = {(ent.start_char, ent.end_char) for ent in doc_sm.ents}
    ref_set = {(s, e) for s, e, _ in ref_spans[i]}
    for ent in suggestions:
        s, e = int(ent["start"]), int(ent["end"])
        reasons = []
        if ent["score"] < SCORE_MIN:
            reasons.append(f"skor {ent['score']:.2f}")
        if (s, e) not in sm_spans:
            reasons.append("spaCy beda")
        if tokens_doc.char_span(s, e) is None:
            reasons.append("tak pas token")
        rows.append({"sent": i, "span": (s, e), "text": text[s:e], "label": ent["entity_group"],
                     "score": float(ent["score"]), "reasons": reasons, "in_ref": (s, e) in ref_set})

for r in rows:
    status = "terima" if not r["reasons"] else "review: " + ", ".join(r["reasons"])
    print(f"{r['text']!r:<22} {r['label']:<5} {r['score']:.2f}  {status:<42} referensi: {'ya' if r['in_ref'] else 'tidak'}")

accepted = [r for r in rows if not r["reasons"]]
found = {(r["sent"], r["span"]) for r in rows if r["in_ref"]}
n_ref = sum(len(spans) for spans in ref_spans)
print()
print(f"{len(rows)} saran: {len(accepted)} diterima otomatis, "
      f"{sum(r['in_ref'] for r in accepted)} di antaranya persis cocok referensi")
print(f"{n_ref} span referensi: {n_ref - len(found)} tidak pernah disarankan dengan batas yang persis sama")

del ner, ner_model
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

'Apple Inc'            ORG   1.00  review: spaCy beda, tak pas token          referensi: tidak
'Austin'               LOC   1.00  terima                                     referensi: ya
'Angela Merkel'        PER   1.00  terima                                     referensi: ya
'Emmanuel Macron'      PER   1.00  terima                                     referensi: ya
'Berlin'               LOC   1.00  terima                                     referensi: ya
'Goldman Sachs'        ORG   1.00  terima                                     referensi: ya
'Brazil'               LOC   1.00  terima                                     referensi: ya
'Steve Wozniak'        PER   1.00  terima                                     referensi: ya
'Steve Jobs'           PER   1.00  terima                                     referensi: ya
'Apple'                ORG   1.00  terima                                     referensi: ya
'Cupertino'            LOC   1.00  terima                                    

Angka tabel ini baru terlihat saat Anda menjalankannya; kami belum menjalankan model ini di GPU. Tiga hal untuk membacanya:

- Model ini **tidak bisa** melabeli `last spring` sebagai DATE atau `$2 billion` sebagai MONEY, karena skema CoNLL-2003 tidak punya label itu; paling jauh ia menyebutnya MISC. Entitas di luar skema model tetap perlu dianotasi manusia atau model lain.
- Skema yang berbeda perlu dipetakan. CoNLL memakai `LOC` untuk kota dan negara, sedangkan spaCy (skema OntoNotes) memisahkan `GPE` dan `LOC`; karena itu perbandingan dengan spaCy di sini hanya melihat posisi span.
- Lolos routing belum tentu benar. Dua model bisa sepakat pada span yang sama-sama salah; kesepakatan antar-model menunjukkan konsistensi, bukan kebenaran. Kolom referensi memperlihatkan selisihnya. Di dunia nyata, perannya diisi item gold yang disisipkan dan audit sampel acak atas saran yang diterima otomatis.

## Menskalakan proses anotasi

Buku menyebut dua strategi. **Pemrosesan terdistribusi** (Dask, PySpark) memparalelkan pipeline anotasi lintas core atau cluster; notebook ini tidak menjalankannya. **Active learning** membalik urutan kerja: latih model kecil, minta ia menunjuk contoh yang paling ia ragukan, labeli contoh itu, lalu ulangi.

> **Listing buku: active learning dengan modAL**
>
> ```python
> learner = ActiveLearner(estimator=RandomForestClassifier(),
>                         X_training=X_pool[:10], y_training=np.random.randint(0, 2, 10))
> query_idx, query_inst = learner.query(X_pool)
> learner.teach(X_pool[query_idx], np.random.randint(0, 2, 1))
> ```
>
> Dengan scikit-learn versi baru, `teach()` gagal: `TypeError: check_X_y() got an unexpected keyword argument 'force_all_finite'`. modAL masih memakai nama argumen lama yang di scikit-learn sudah diganti `ensure_all_finite`. Error ini tercatat di notebook asli (scikit-learn 1.8.0) dan kami ulangi secara lokal (scikit-learn 1.9.1, modAL-python 0.4.2.1, versi terbaru di PyPI saat diuji). Seandainya berjalan pun, fitur dan labelnya acak, sehingga skornya akan berkisar 0,5 dan tidak membuktikan apa pun. Versi kerjanya di bawah menulis uncertainty sampling sendiri.

### Under the hood: active learning vs sampling acak

Datanya `load_digits` dari scikit-learn: 1.797 gambar angka 8×8 dengan 10 kelas, tersedia offline. Sebanyak 500 gambar menjadi data uji, sisanya pool "belum berlabel". Label asli dataset berperan sebagai annotator *oracle* yang selalu benar, jadi yang diukur adalah nilai *pemilihan* contoh, bukan mutu annotator. Kedua strategi mulai dari 20 label acak yang sama dan menambah 10 label per putaran: strategi acak memilih sembarang, uncertainty sampling memilih contoh dengan **margin** terkecil, yaitu selisih peluang kelas teratas dan kelas kedua. Modelnya regresi logistik di CPU, diulang untuk 5 seed.

In [28]:
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression

X, y = load_digits(return_X_y=True)
X = X / 16.0
rng0 = np.random.default_rng(0)
test_idx = rng0.choice(len(X), size=500, replace=False)
pool_idx = np.setdiff1d(np.arange(len(X)), test_idx)
X_test, y_test = X[test_idx], y[test_idx]
BUDGETS = [20, 50, 100, 200, 400]


def run_strategy(seed, active):
    rng = np.random.default_rng(seed)
    labeled = list(rng.choice(pool_idx, size=20, replace=False))
    labeled_set = set(labeled)
    unlabeled = [i for i in pool_idx if i not in labeled_set]
    accs = {}
    while True:
        clf = LogisticRegression(max_iter=2000).fit(X[labeled], y[labeled])
        if len(labeled) in BUDGETS:
            accs[len(labeled)] = clf.score(X_test, y_test)
        if len(labeled) >= BUDGETS[-1]:
            return accs
        if active:
            proba = clf.predict_proba(X[unlabeled])
            top2 = np.sort(proba, axis=1)[:, -2:]
            pick = np.argsort(top2[:, 1] - top2[:, 0])[:10]   # margin terkecil = paling ragu
        else:
            pick = rng.choice(len(unlabeled), size=10, replace=False)
        for j in sorted(pick, reverse=True):
            labeled.append(unlabeled.pop(j))


results = {"acak": {b: [] for b in BUDGETS}, "uncertainty": {b: [] for b in BUDGETS}}
for seed in range(5):
    for name, active in (("acak", False), ("uncertainty", True)):
        for budget, acc in run_strategy(seed, active).items():
            results[name][budget].append(acc)

print(f"{'label':>6} | {'acak (rata2 ± sd, 5 seed)':>26} | {'uncertainty (rata2 ± sd)':>25}")
for b in BUDGETS:
    r = np.array(results["acak"][b])
    a = np.array(results["uncertainty"][b])
    print(f"{b:>6} | {r.mean():>18.3f} ± {r.std():.3f} | {a.mean():>17.3f} ± {a.std():.3f}")

 label |  acak (rata2 ± sd, 5 seed) |  uncertainty (rata2 ± sd)
    20 |              0.601 ± 0.025 |             0.601 ± 0.025
    50 |              0.797 ± 0.028 |             0.854 ± 0.027
   100 |              0.865 ± 0.027 |             0.920 ± 0.006
   200 |              0.919 ± 0.004 |             0.950 ± 0.003
   400 |              0.938 ± 0.006 |             0.959 ± 0.001


Baris 20 identik karena kedua strategi mulai dari 20 label yang sama. Setelah itu uncertainty sampling unggul di setiap anggaran. Bacalah secara mendatar: sampling acak butuh 200 label untuk mencapai 0,919, sedangkan uncertainty sampling mencapai 0,920 dengan 100 label. Di tengah kurva, satu label pilihan model bernilai kira-kira dua label acak. Selisihnya pada 100 label (0,920 vs 0,865) sekitar dua kali simpangan baku antar-seed yang terbesar (0,027), jadi bukan kebetulan satu seed.

Keunggulannya menyempit dari 5,7 poin persentase di 50 label menjadi 2,1 poin di 400 label, karena kedua kurva mendekati plafon model yang sama: active learning mempercepat jalan ke plafon, tidak meninggikannya. Hasil ini berlaku untuk digit yang bersih dengan annotator oracle. Pada data nyata, contoh yang paling "meragukan" sering juga outlier atau item yang labelnya rancu, jadi uncertainty sampling biasanya dipadukan dengan diversity sampling dan porsi acak sebagai kontrol.

## Bias anotasi dan mitigasinya

Katalog bias di buku, masing-masing dengan penangkalnya:

- **Selection bias**, data yang dianotasi tidak mewakili data produksi: sampling representatif, oversampling kelompok yang kurang terwakili.
- **Labeling bias**, interpretasi subjektif atau kultural annotator: pedoman rinci dengan contoh dan kasus tepi, tim annotator yang beragam.
- **Confirmation bias**, melabeli sesuai dugaan awal: sesi kalibrasi rutin, semua orang melabeli item yang sama lalu membahas selisihnya.
- **Automation bias**, menerima saran model tanpa memeriksa: item gold yang disisipkan dan audit tingkat koreksi per annotator.
- **Pedoman yang ambigu**, sumber noise yang paling murah diperbaiki: revisi pedoman dari umpan balik, lalu ukur dampaknya lewat IAA.

Benang merahnya: IAA yang rendah adalah alarm bahwa pedoman, pelatihan, atau definisi tugasnya bermasalah, bukan sekadar tanda annotator yang buruk. Mitigasi bias berjalan sepanjang umur dataset, bukan satu pemeriksaan di akhir.

## Ringkasan

- **Span yang ditolak harus bersuara.** Listing buku menyimpan kalimat pertama tanpa entitas karena `if span:` menelan `None`. Turunkan offset dari teks entitas, laporkan setiap span yang ditolak, dan baca balik file-nya. Meleset ±1 karakter saja membuat `strict` menolak 80% span; `expand` menyelamatkan semuanya, tetapi pada ±3 hanya 52,9% hasilnya yang masih tepat.
- **Label kata tidak sama dengan wordpiece.** `zip` menggeser 7 label di atas 8 wordpiece; `word_ids()` dan `offset_mapping` adalah jembatan yang benar, dan label yang sudah selaras bisa langsung dipakai melatih head BERT di GPU.
- **Kesepakatan mentah menipu pada data timpang.** Pada 95% token `O`, dua penebak sepakat 90,4% sedangkan dua annotator 90% hanya 82,3%; kappa (sekitar 0 vs 0,26) yang membedakan keduanya.
- **Annotator tambahan membantu dengan imbal hasil yang menurun**, dan majority vote polos bisa kalah dari satu ahli (0,787 vs 0,901); bobot reliabilitas mencegahnya bila reliabilitas bisa diestimasi.
- **Pra-anotasi dan active learning menghemat label tetapi tetap perlu kontrol.** Saran model butuh routing dan audit, karena kesepakatan antar-model bukan bukti kebenaran. Di data digit, uncertainty sampling menyamai akurasi sampling acak dengan sekitar separuh label.

Bab berikutnya di buku membahas *training pipeline*, tempat data beranotasi seperti ini dipakai untuk melatih model.